# Import Data & Functions

In [1]:
import pandas as pd
import numpy as np

# CSV 파일 읽기
df = pd.read_csv('https://raw.githubusercontent.com/sw1kwon/usa-elections/refs/heads/main/clean/Presidential_Elections/1976-2020-president_v1.csv')

In [2]:
df.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,candidate,party_detailed,writein,candidatevotes,totalvotes,version,notes,party_simplified
0,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"CARTER, JIMMY",DEMOCRAT,False,659170,1182850,20210113,NaN,DEMOCRAT
1,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"FORD, GERALD",REPUBLICAN,False,504070,1182850,20210113,NaN,REPUBLICAN
2,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"MADDOX, LESTER",AMERICAN INDEPENDENT PARTY,False,9198,1182850,20210113,NaN,OTHER
3,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"BUBAR, BENJAMIN """"BEN""""",PROHIBITION,False,6669,1182850,20210113,NaN,OTHER
4,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"HALL, GUS",COMMUNIST PARTY USE,False,1954,1182850,20210113,NaN,OTHER


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4287 entries, 0 to 4286
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              4287 non-null   int64  
 1   state             4287 non-null   object 
 2   state_po          4287 non-null   object 
 3   state_fips        4287 non-null   int64  
 4   state_cen         4287 non-null   int64  
 5   state_ic          4287 non-null   int64  
 6   office            4287 non-null   object 
 7   candidate         4000 non-null   object 
 8   party_detailed    3831 non-null   object 
 9   writein           4287 non-null   bool   
 10  candidatevotes    4287 non-null   int64  
 11  totalvotes        4287 non-null   int64  
 12  version           4287 non-null   int64  
 13  notes             0 non-null      float64
 14  party_simplified  4287 non-null   object 
dtypes: bool(1), float64(1), int64(7), object(6)
memory usage: 473.2+ KB


In [4]:
df['party_simplified'].value_counts()

,count
party_simplified,
OTHER,2524
DEMOCRAT,615
REPUBLICAN,613
LIBERTARIAN,535


In [5]:
import pandas as pd
import numpy as np


def analyze_vote_aggregation(df, year):
    """
    정당별 득표수를 집계하여 미국 대통령 선거 데이터를 분석하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        미국 선거 데이터 (1976-2020)
    year : int
        분석할 선거 연도 (1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020)

    Returns:
    --------
    pandas.DataFrame
        주별 득표수 집계 분석 결과
    """

    # 연도 유효성 검사
    valid_years = list(range(1976, 2021, 4))

    if year not in valid_years:
        raise ValueError(f"유효하지 않은 연도입니다. 가능한 연도: {valid_years}")

    # 해당 연도 데이터가 존재하는지 확인
    if year not in df['year'].values:
        raise ValueError(f"{year}년 데이터가 데이터셋에 존재하지 않습니다.")

    # 메서드 체이닝으로 데이터 처리
    result = (df
        # 지정된 연도 데이터만 필터링
        .query(f'year == {year}')

        # writein 컬럼의 NA 값 처리
        .assign(writein=lambda x: x['writein'].fillna(True))

        # 주별로 그룹화하여 득표수 집계
        .groupby('state')
        .apply(lambda group: pd.Series({
            'election_year': group['year'].iloc[0],
            'state_code': group['state_po'].iloc[0],
            'total_votes': group['totalvotes'].iloc[0],

            # writein=FALSE인 정당별 득표수 합계
            'votes_republican': group.query('writein == False & party_simplified == "REPUBLICAN"')['candidatevotes'].sum(),
            'votes_democrat': group.query('writein == False & party_simplified == "DEMOCRAT"')['candidatevotes'].sum(),
            'votes_libertarian': group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidatevotes'].sum(),
            'votes_other': group.query('writein == False & party_simplified == "OTHER"')['candidatevotes'].sum(),

            # 모든 기명투표 득표수 합계
            'votes_writein': group.query('writein == True')['candidatevotes'].sum()
        }), include_groups=False)

        # 인덱스를 컬럼으로 변환
        .reset_index()

        # 주 이름 기준으로 정렬
        .sort_values('state')

        # 컬럼 선택 및 순서 정렬
        [['election_year', 'state', 'state_code', 'total_votes', 'votes_republican',
          'votes_democrat', 'votes_libertarian', 'votes_other', 'votes_writein']]

        # 인덱스 재설정
        .reset_index(drop=True)

        .rename(columns={
        "election_year": "year",
        "state_code": "state_po",
        "total_votes": "totalvotes"
        })
    )

    return result

def process_all_years_vote_aggregation(df, years=None):
    """
    여러 연도의 득표수 집계를 처리하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        미국 선거 데이터
    years : list, optional
        처리할 연도 리스트. None이면 모든 가능한 연도를 처리

    Returns:
    --------
    pandas.DataFrame
        모든 연도의 결과를 합친 데이터프레임
    """

    if years is None:
        # 데이터셋에 존재하는 유효한 연도들 가져오기
        valid_years = list(range(1976, 2021, 4))
        existing_years = df['year'].unique()
        years = [year for year in valid_years if year in existing_years]

    # 각 연도별로 처리
    results = []
    for year in years:
        try:
            result = analyze_vote_aggregation(df, year)
            results.append(result)
            print(f"{year}년 분석 완료: {len(result)}개 주/지역")
        except Exception as e:
            print(f"{year}년 분석 실패: {e}")

    # 모든 결과 합치기
    if results:
        combined_result = pd.concat(results, ignore_index=True)
        return combined_result
    else:
        return pd.DataFrame()

### 사용 예시
# result_1976 = analyze_vote_aggregation(df, 1976)
# result_multi = process_all_years_vote_aggregation(df, [1976, 1980, 1984])
# result_all = process_all_years_vote_aggregation(df)  # 모든 연도

In [6]:
import pandas as pd
import numpy as np

def analyze_election_data(df, year):
    """
    미국 대통령 선거 데이터를 분석하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        미국 선거 데이터 (1976-2020)
    year : int
        분석할 선거 연도 (1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020)

    Returns:
    --------
    pandas.DataFrame
        주별 선거 결과 분석 데이터

    Raises:
    -------
    ValueError
        유효하지 않은 연도가 입력된 경우
    """

    # 유효한 연도 확인 (1976부터 2020까지 4년 간격)
    valid_years = list(range(1976, 2021, 4))

    if year not in valid_years:
        raise ValueError(f"유효하지 않은 연도입니다. 가능한 연도: {valid_years}")

    # 해당 연도 데이터가 실제로 존재하는지 확인
    if year not in df['year'].values:
        raise ValueError(f"{year}년 데이터가 데이터셋에 존재하지 않습니다.")

    # 메서드 체이닝으로 데이터 처리
    result = (df
        # 지정된 연도 데이터만 필터링
        .query(f'year == {year}')

        # state별로 그룹화하여 각종 집계 수행
        .groupby('state')
        .apply(lambda group: pd.Series({
            'year': group['year'].iloc[0],
            'state_po': group['state_po'].iloc[0],

            # 득표수 기준으로 정렬하여 1위, 2위 구하기
            'first_place_party': group.nlargest(1, 'candidatevotes')['party_simplified'].iloc[0],
            'first_place_vote_share': round(group.nlargest(1, 'candidatevotes')['candidatevotes'].iloc[0] /
                                     group.nlargest(1, 'candidatevotes')['totalvotes'].iloc[0], 4),
            'second_place_party': group.nlargest(2, 'candidatevotes')['party_simplified'].iloc[1] if len(group) > 1 else '',
            'second_place_vote_share': round(group.nlargest(2, 'candidatevotes')['candidatevotes'].iloc[1] /
                                      group.nlargest(2, 'candidatevotes')['totalvotes'].iloc[1] if len(group) > 1 else 0, 4),

            # writein이 FALSE인 경우의 정당별 후보자 수
            'republican_candidates': len(group.query('writein == False & party_simplified == "REPUBLICAN"')),
            'democrat_candidates': len(group.query('writein == False & party_simplified == "DEMOCRAT"')),
            'libertarian_candidates': len(group.query('writein == False & party_simplified == "LIBERTARIAN"')),
            'other_candidates': len(group.query('writein == False & party_simplified == "OTHER"')),

            # writein이 TRUE인 경우의 정당별 후보자 수
            'republican_candidates_writein': len(group.query('writein == True & party_simplified == "REPUBLICAN"')),
            'democrat_candidates_writein': len(group.query('writein == True & party_simplified == "DEMOCRAT"')),
            'libertarian_candidates_writein': len(group.query('writein == True & party_simplified == "LIBERTARIAN"')),
            'other_candidates_writein': len(group.query('writein == True & party_simplified == "OTHER"'))
        }), include_groups=False)

        # 인덱스를 state 컬럼으로 변환
        .reset_index()

        # state 기준으로 정렬
        .sort_values('state')

        # 필요한 컬럼만 선택하고 순서 정렬
        [['year', 'state', 'state_po', 'first_place_party', 'first_place_vote_share',
          'second_place_party', 'second_place_vote_share', 'republican_candidates',
          'democrat_candidates', 'libertarian_candidates', 'other_candidates',
          'republican_candidates_writein', 'democrat_candidates_writein',
          'libertarian_candidates_writein', 'other_candidates_writein']]

        # 인덱스 재설정
        .reset_index(drop=True)
    )

    return result

def analyze_multiple_elections(df, years=None):
    """
    여러 년도의 선거 데이터를 한번에 분석하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        미국 선거 데이터
    years : list, optional
        분석할 연도 리스트. None이면 모든 가능한 연도를 분석

    Returns:
    --------
    pandas.DataFrame
        모든 연도의 분석 결과를 합친 데이터프레임
    """

    if years is None:
        # 데이터에 실제로 존재하는 연도 중 유효한 연도만 선택
        valid_years = list(range(1976, 2021, 4))
        existing_years = df['year'].unique()
        years = [year for year in valid_years if year in existing_years]

    # 각 연도별로 분석 실행
    results = []
    for year in years:
        try:
            result = analyze_election_data(df, year)
            results.append(result)
            print(f"{year}년 분석 완료: {len(result)}개 주/지역")
        except Exception as e:
            print(f"{year}년 분석 실패: {e}")

    # 모든 결과 합치기
    if results:
        combined_result = pd.concat(results, ignore_index=True)
        return combined_result
    else:
        return pd.DataFrame()

### 함수 사용법 안내
# result_1976 = analyze_election_data(df, 1976)
# result_multi = analyze_multiple_elections(df, [1976, 1980, 1984])
# result_all = analyze_multiple_elections(df)  # 모든 연도

# temp1

## csv

In [7]:
# 1976부터 2020까지 4년 간격으로 반복
years = list(range(1976, 2021, 4))

for year in years:
    try:
        print(f"{year}년 데이터 처리 중...")

        # 기존 함수 사용해서 분석
        result = analyze_vote_aggregation(df, year)

        # 파일명 생성 및 저장
        filename = f'temp1_president_{year}.csv'
        result.to_csv(filename, encoding='utf-8-sig', index=False)

        print(f"{year}년 완료: {filename} 저장됨 ({len(result)}개 주/지역)")

    except Exception as e:
        print(f"{year}년 처리 실패: {e}")

print("\n모든 연도 처리 완료!")

1976년 데이터 처리 중...
1976년 완료: temp1_president_1976.csv 저장됨 (51개 주/지역)
1980년 데이터 처리 중...
1980년 완료: temp1_president_1980.csv 저장됨 (51개 주/지역)
1984년 데이터 처리 중...
1984년 완료: temp1_president_1984.csv 저장됨 (51개 주/지역)
1988년 데이터 처리 중...
1988년 완료: temp1_president_1988.csv 저장됨 (51개 주/지역)
1992년 데이터 처리 중...
1992년 완료: temp1_president_1992.csv 저장됨 (51개 주/지역)
1996년 데이터 처리 중...
1996년 완료: temp1_president_1996.csv 저장됨 (51개 주/지역)
2000년 데이터 처리 중...
2000년 완료: temp1_president_2000.csv 저장됨 (51개 주/지역)
2004년 데이터 처리 중...
2004년 완료: temp1_president_2004.csv 저장됨 (51개 주/지역)
2008년 데이터 처리 중...
2008년 완료: temp1_president_2008.csv 저장됨 (51개 주/지역)
2012년 데이터 처리 중...
2012년 완료: temp1_president_2012.csv 저장됨 (51개 주/지역)
2016년 데이터 처리 중...
2016년 완료: temp1_president_2016.csv 저장됨 (51개 주/지역)
2020년 데이터 처리 중...
2020년 완료: temp1_president_2020.csv 저장됨 (51개 주/지역)

모든 연도 처리 완료!


# tidy1

## csv

In [8]:
result_all1 = process_all_years_vote_aggregation(df)

1976년 분석 완료: 51개 주/지역
1980년 분석 완료: 51개 주/지역
1984년 분석 완료: 51개 주/지역
1988년 분석 완료: 51개 주/지역
1992년 분석 완료: 51개 주/지역
1996년 분석 완료: 51개 주/지역
2000년 분석 완료: 51개 주/지역
2004년 분석 완료: 51개 주/지역
2008년 분석 완료: 51개 주/지역
2012년 분석 완료: 51개 주/지역
2016년 분석 완료: 51개 주/지역
2020년 분석 완료: 51개 주/지역


In [9]:
result_all1.to_csv("tidy1_president.csv", index=False, encoding="utf-8-sig")

# temp2

## csv

In [10]:
# 1976부터 2020까지 4년 간격으로 반복
years = list(range(1976, 2021, 4))

for year in years:
    try:
        print(f"{year}년 데이터 처리 중...")

        # 기존 함수 사용해서 분석
        result = analyze_election_data(df, year)

        # 파일명 생성 및 저장
        filename = f'temp2_president_{year}.csv'
        result.to_csv(filename, encoding='utf-8-sig', index=False)

        print(f"{year}년 완료: {filename} 저장됨 ({len(result)}개 주/지역)")

    except Exception as e:
        print(f"{year}년 처리 실패: {e}")

print("\n모든 연도 처리 완료!")

1976년 데이터 처리 중...
1976년 완료: temp2_president_1976.csv 저장됨 (51개 주/지역)
1980년 데이터 처리 중...
1980년 완료: temp2_president_1980.csv 저장됨 (51개 주/지역)
1984년 데이터 처리 중...
1984년 완료: temp2_president_1984.csv 저장됨 (51개 주/지역)
1988년 데이터 처리 중...
1988년 완료: temp2_president_1988.csv 저장됨 (51개 주/지역)
1992년 데이터 처리 중...
1992년 완료: temp2_president_1992.csv 저장됨 (51개 주/지역)
1996년 데이터 처리 중...
1996년 완료: temp2_president_1996.csv 저장됨 (51개 주/지역)
2000년 데이터 처리 중...
2000년 완료: temp2_president_2000.csv 저장됨 (51개 주/지역)
2004년 데이터 처리 중...
2004년 완료: temp2_president_2004.csv 저장됨 (51개 주/지역)
2008년 데이터 처리 중...
2008년 완료: temp2_president_2008.csv 저장됨 (51개 주/지역)
2012년 데이터 처리 중...
2012년 완료: temp2_president_2012.csv 저장됨 (51개 주/지역)
2016년 데이터 처리 중...
2016년 완료: temp2_president_2016.csv 저장됨 (51개 주/지역)
2020년 데이터 처리 중...
2020년 완료: temp2_president_2020.csv 저장됨 (51개 주/지역)

모든 연도 처리 완료!


# tidy2

## csv

In [11]:
result_all2 = analyze_multiple_elections(df)

1976년 분석 완료: 51개 주/지역
1980년 분석 완료: 51개 주/지역
1984년 분석 완료: 51개 주/지역
1988년 분석 완료: 51개 주/지역
1992년 분석 완료: 51개 주/지역
1996년 분석 완료: 51개 주/지역
2000년 분석 완료: 51개 주/지역
2004년 분석 완료: 51개 주/지역
2008년 분석 완료: 51개 주/지역
2012년 분석 완료: 51개 주/지역
2016년 분석 완료: 51개 주/지역
2020년 분석 완료: 51개 주/지역


In [12]:
result_all2.to_csv("tidy2_president.csv", index=False, encoding="utf-8-sig")

## EDA

In [13]:
result_all2.head()

,year,state,state_po,first_place_party,first_place_vote_share,second_place_party,second_place_vote_share,republican_candidates,democrat_candidates,libertarian_candidates,other_candidates,republican_candidates_writein,democrat_candidates_writein,libertarian_candidates_writein,other_candidates_writein
0,1976,ALABAMA,AL,DEMOCRAT,0.5573,REPUBLICAN,0.4261,1,1,1,3,0,0,0,1
1,1976,ALASKA,AK,REPUBLICAN,0.5790,DEMOCRAT,0.3565,1,1,1,0,0,0,0,1
2,1976,ARIZONA,AZ,REPUBLICAN,0.5637,DEMOCRAT,0.3980,1,1,1,4,0,0,0,1
3,1976,ARKANSAS,AR,DEMOCRAT,0.6496,REPUBLICAN,0.3490,1,1,0,2,0,0,0,0
4,1976,CALIFORNIA,CA,REPUBLICAN,0.4975,DEMOCRAT,0.4795,1,1,0,5,0,0,0,0


In [14]:
result_all2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   year                            612 non-null    int64  
 1   state                           612 non-null    object 
 2   state_po                        612 non-null    object 
 3   first_place_party               612 non-null    object 
 4   first_place_vote_share          612 non-null    float64
 5   second_place_party              612 non-null    object 
 6   second_place_vote_share         612 non-null    float64
 7   republican_candidates           612 non-null    int64  
 8   democrat_candidates             612 non-null    int64  
 9   libertarian_candidates          612 non-null    int64  
 10  other_candidates                612 non-null    int64  
 11  republican_candidates_writein   612 non-null    int64  
 12  democrat_candidates_writein     612 

In [15]:
result_all2['first_place_party'].value_counts()

,count
first_place_party,
REPUBLICAN,358
DEMOCRAT,254


In [16]:
result_all2['second_place_party'].value_counts()

,count
second_place_party,
DEMOCRAT,357
REPUBLICAN,253
OTHER,2


- https://en.wikipedia.org/wiki/1992_United_States_presidential_election_in_Maine
- https://en.wikipedia.org/wiki/1992_United_States_presidential_election_in_Utah

In [17]:
result_all2.loc[result_all2['second_place_party'] == 'OTHER']

,year,state,state_po,first_place_party,first_place_vote_share,second_place_party,second_place_vote_share,republican_candidates,democrat_candidates,libertarian_candidates,other_candidates,republican_candidates_writein,democrat_candidates_writein,libertarian_candidates_writein,other_candidates_writein
223,1992,MAINE,ME,DEMOCRAT,0.3877,OTHER,0.3044,1,1,1,4,0,0,0,0
248,1992,UTAH,UT,REPUBLICAN,0.4336,OTHER,0.2734,1,1,1,10,0,0,0,0


In [18]:
cols = [
    "republican_candidates",
    "democrat_candidates",
    "libertarian_candidates",
    "other_candidates",
    "republican_candidates_writein",
    "democrat_candidates_writein",
    "libertarian_candidates_writein",
    "other_candidates_writein"
]

for col in cols:
    print(f"\n=== {col} ===")
    print(result_all2[col].value_counts(dropna=False))



=== republican_candidates ===
republican_candidates
1    612
Name: count, dtype: int64

=== democrat_candidates ===
democrat_candidates
1    612
Name: count, dtype: int64

=== libertarian_candidates ===
libertarian_candidates
1    535
0     77
Name: count, dtype: int64

=== other_candidates ===
other_candidates
2     115
1     112
3     108
4      73
5      49
0      44
6      38
7      32
8      21
10     10
9       4
13      2
20      2
11      1
17      1
Name: count, dtype: int64

=== republican_candidates_writein ===
republican_candidates_writein
0    611
1      1
Name: count, dtype: int64

=== democrat_candidates_writein ===
democrat_candidates_writein
0    609
1      3
Name: count, dtype: int64

=== libertarian_candidates_writein ===
libertarian_candidates_writein
0    612
Name: count, dtype: int64

=== other_candidates_writein ===
other_candidates_writein
0     346
1     235
2       7
5       5
4       3
10      2
17      2
6       2
9       2
8       2
3       1
16      1
14 

In [19]:
df.loc[(df['writein'] == True) & (df['party_simplified'] == "DEMOCRAT")]

,year,state,state_po,state_fips,state_cen,state_ic,office,candidate,party_detailed,writein,candidatevotes,totalvotes,version,notes,party_simplified
2544,2004,MARYLAND,MD,24,52,52,US PRESIDENT,OTHER,DEMOCRAT,True,7,2384238,20210113,NaN,DEMOCRAT
3414,2016,ARIZONA,AZ,4,86,61,US PRESIDENT,NaN,DEMOCRAT,True,42,2573165,20210113,NaN,DEMOCRAT
3543,2016,MARYLAND,MD,24,52,52,US PRESIDENT,"CLINTON, HILLARY",DEMOCRAT,True,78,2781446,20210113,NaN,DEMOCRAT


In [20]:
df.loc[(df['writein'] == True) & (df['party_simplified'] == "REPUBLICAN")]

,year,state,state_po,state_fips,state_cen,state_ic,office,candidate,party_detailed,writein,candidatevotes,totalvotes,version,notes,party_simplified
3542,2016,MARYLAND,MD,24,52,52,US PRESIDENT,"TRUMP, DONALD J.",REPUBLICAN,True,259,2781446,20210113,NaN,REPUBLICAN


# Batch CSV Files to ZIP

In [21]:
import zipfile
import glob

# Find all CSV files in current directory
csv_files = glob.glob('*.csv')

# Create ZIP file
with zipfile.ZipFile('all_csv_files.zip', 'w') as zipf:
   for file in csv_files:
       zipf.write(file)
       print(f"Added: {file}")  # Show progress

print(f"Total {len(csv_files)} files compressed.")

Added: temp1_president_1996.csv
Added: tidy2_president.csv
Added: temp1_president_2008.csv
Added: temp2_president_2016.csv
Added: temp2_president_2008.csv
Added: temp2_president_2012.csv
Added: temp1_president_1984.csv
Added: temp1_president_2016.csv
Added: temp1_president_1980.csv
Added: temp1_president_2020.csv
Added: temp2_president_1976.csv
Added: temp1_president_2012.csv
Added: temp2_president_1996.csv
Added: temp2_president_2020.csv
Added: temp2_president_1984.csv
Added: temp1_president_1988.csv
Added: temp2_president_2004.csv
Added: tidy1_president.csv
Added: temp1_president_1976.csv
Added: temp1_president_2000.csv
Added: temp2_president_1980.csv
Added: temp1_president_1992.csv
Added: temp2_president_1992.csv
Added: temp2_president_1988.csv
Added: temp1_president_2004.csv
Added: temp2_president_2000.csv
Total 26 files compressed.
